In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


file paths:('/content/drive/MyDrive/THESIS/05outputs/partial_results.csv')
('/content/drive/MyDrive/THESIS/04data/customer-churn.csv')

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [3]:
file_path = '/content/drive/MyDrive/THESIS/04data/customer-churn.csv'

try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded {file_path}:")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully loaded /content/drive/MyDrive/THESIS/04data/customer-churn.csv:


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [4]:
print("Shape of df:")
print(df.shape)

print("\nNull values in df_filtered:")
print(df.isnull().sum())

print("\nData types of df_filtered:")
print(df.dtypes)

Shape of df:
(7043, 33)

Null values in df_filtered:
CustomerID              0
Count                   0
Country                 0
State                   0
City                    0
Zip Code                0
Lat Long                0
Latitude                0
Longitude               0
Gender                  0
Senior Citizen          0
Partner                 0
Dependents              0
Tenure Months           0
Phone Service           0
Multiple Lines          0
Internet Service        0
Online Security         0
Online Backup           0
Device Protection       0
Tech Support            0
Streaming TV            0
Streaming Movies        0
Contract                0
Paperless Billing       0
Payment Method          0
Monthly Charges         0
Total Charges           0
Churn Label             0
Churn Value             0
Churn Score             0
CLTV                    0
Churn Reason         5174
dtype: int64

Data types of df_filtered:
CustomerID            object
Count              

In [5]:
df_original = df.copy()
print("A copy of the original DataFrame 'df' has been created as 'df_original'.")
print(f"Shape of df_original: {df_original.shape}")

A copy of the original DataFrame 'df' has been created as 'df_original'.
Shape of df_original: (7043, 33)


In [6]:
import pandas as pd

# Define columns to drop from the original df to create df_filtered
columns_to_drop = [
    'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Churn Label'
]

# Create df_filtered by dropping specified columns from df
df_filtered = df.drop(columns=columns_to_drop).copy()

df_filtered['Total Charges'] = pd.to_numeric(df_filtered['Total Charges'], errors='coerce')
df_filtered.dropna(subset=['Total Charges'], inplace=True)

print("Shape of df_filtered after dropping columns and cleaning 'Total Charges':")
print(df_filtered.shape)

print("\nNull values in df_filtered after cleaning:")
print(df_filtered.isnull().sum())

print("\nData types of df_filtered after cleaning:")
print(df_filtered.dtypes)

print("\nHead of df_filtered:")
display(df_filtered.head())

Shape of df_filtered after dropping columns and cleaning 'Total Charges':
(7032, 24)

Null values in df_filtered after cleaning:
CustomerID              0
Gender                  0
Senior Citizen          0
Partner                 0
Dependents              0
Tenure Months           0
Phone Service           0
Multiple Lines          0
Internet Service        0
Online Security         0
Online Backup           0
Device Protection       0
Tech Support            0
Streaming TV            0
Streaming Movies        0
Contract                0
Paperless Billing       0
Payment Method          0
Monthly Charges         0
Total Charges           0
Churn Value             0
Churn Score             0
CLTV                    0
Churn Reason         5163
dtype: int64

Data types of df_filtered after cleaning:
CustomerID            object
Gender                object
Senior Citizen        object
Partner               object
Dependents            object
Tenure Months          int64
Phone Service    

,CustomerID,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,...,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,...,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,86,3239,Competitor made better offer
1,9237-HQITU,Female,No,No,Yes,2,Yes,No,Fiber optic,No,...,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,67,2701,Moved
2,9305-CDSKC,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,...,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1,86,5372,Moved
3,7892-POOKP,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,...,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1,84,5003,Moved
4,0280-XJGEX,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,...,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1,89,5340,Competitor had better devices


In [7]:
import pickle
import pandas as pd

# Check if df_filtered exists. If not, reconstruct it from existing 'df' if available, or load from CSV.
if 'df_filtered' not in locals() and 'df_filtered' not in globals():
    print("df_filtered not found in the current session.")

    # Define the list of selected columns as used previously
    selected_columns = [
        'CustomerID', 'Gender', 'Senior Citizen', 'Partner', 'Dependents',
        'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service',
        'Online Security', 'Online Backup', 'Device Protection', 'Tech Support',
        'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing',
        'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Value',
        'Churn Score', 'CLTV', 'Churn Reason'
    ]

columns_to_encode = ['Paperless Billing','Device Protection', 'Online Security', 'Online Backup', 'Multiple Lines', 'Tech Support' ,'Senior Citizen', 'Partner', 'Dependents','Streaming TV', 'Streaming Movies','Phone Service']

# Create a dictionary to store the encoding mappings
encoders = {}

# Define a custom mapping function for 'Yes'/'No' and similar binary columns
def binary_encode(value):
    if value in ['Yes', 'Y']: # Handle potential variations of 'Yes'
        return 1
    elif value in ['No', 'N', 'No phone service', 'No internet service']: # Handle potential variations of 'No'
        return 0
    return value # Return original value if no match, though not expected for these columns

for col in columns_to_encode:
    # Apply the custom binary encoding
    df_filtered[col] = df_filtered[col].apply(binary_encode)
    # Store the mapping (for these simple binary encodings, the function/logic itself is the 'encoder')
    # For simplicity here, we'll note that the encoding is 'Yes':1, 'No':0
    encoders[col] = {'Yes': 1, 'No': 0, 'No phone service': 0, 'No internet service': 0}

print("Columns encoded to 0 (No) and 1 (Yes):")
for col in columns_to_encode:
    print(f"  - {col}: {df_filtered[col].unique()}")
    print(f"    (Original dtypes: {df_filtered[col].dtype})")

# Save the encoders (mappings) using pickle
output_path = '/content/drive/MyDrive/THESIS/05outputs/encoders.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(encoders, f)

print(f"\nEncoders saved to {output_path}")

# Display data types and head to verify
print("\nData types after encoding:")
display(df_filtered[columns_to_encode].dtypes)
print("\nHead of df_filtered showing encoded columns:")
display(df_filtered[columns_to_encode].head())

Columns encoded to 0 (No) and 1 (Yes):
  - Paperless Billing: [1 0]
    (Original dtypes: int64)
  - Device Protection: [0 1]
    (Original dtypes: int64)
  - Online Security: [1 0]
    (Original dtypes: int64)
  - Online Backup: [1 0]
    (Original dtypes: int64)
  - Multiple Lines: [0 1]
    (Original dtypes: int64)
  - Tech Support: [0 1]
    (Original dtypes: int64)
  - Senior Citizen: [0 1]
    (Original dtypes: int64)
  - Partner: [0 1]
    (Original dtypes: int64)
  - Dependents: [0 1]
    (Original dtypes: int64)
  - Streaming TV: [0 1]
    (Original dtypes: int64)
  - Streaming Movies: [0 1]
    (Original dtypes: int64)
  - Phone Service: [1 0]
    (Original dtypes: int64)

Encoders saved to /content/drive/MyDrive/THESIS/05outputs/encoders.pkl

Data types after encoding:


,0
Paperless Billing,int64
Device Protection,int64
Online Security,int64
Online Backup,int64
Multiple Lines,int64
Tech Support,int64
Senior Citizen,int64
Partner,int64
Dependents,int64
Streaming TV,int64



Head of df_filtered showing encoded columns:


,Paperless Billing,Device Protection,Online Security,Online Backup,Multiple Lines,Tech Support,Senior Citizen,Partner,Dependents,Streaming TV,Streaming Movies,Phone Service
0,1,0,1,1,0,0,0,0,0,0,0,1
1,1,0,0,0,0,0,0,0,1,0,0,1
2,1,1,0,0,1,0,0,0,1,1,1,1
3,1,1,0,0,1,1,0,1,1,1,1,1
4,1,1,0,1,1,0,0,0,1,1,1,1


In [8]:
for col in df_filtered :
  print(f'({col}: {df_filtered[col].unique()})')

(CustomerID: ['3668-QPYBK' '9237-HQITU' '9305-CDSKC' ... '2234-XADUH' '4801-JZAZL'
 '3186-AJIEK'])
(Gender: ['Male' 'Female'])
(Senior Citizen: [0 1])
(Partner: [0 1])
(Dependents: [0 1])
(Tenure Months: [ 2  8 28 49 10  1 47 17  5 34 11 15 18  9  7 12 25 68 55 37  3 27 20  4
 58 53 13  6 19 59 16 52 24 32 38 54 43 63 21 69 22 61 60 48 40 23 39 35
 56 65 33 30 45 46 62 70 50 44 71 26 14 41 66 64 29 42 67 51 31 57 36 72])
(Phone Service: [1 0])
(Multiple Lines: [0 1])
(Internet Service: ['DSL' 'Fiber optic' 'No'])
(Online Security: [1 0])
(Online Backup: [1 0])
(Device Protection: [0 1])
(Tech Support: [0 1])
(Streaming TV: [0 1])
(Streaming Movies: [0 1])
(Contract: ['Month-to-month' 'Two year' 'One year'])
(Paperless Billing: [1 0])
(Payment Method: ['Mailed check' 'Electronic check' 'Bank transfer (automatic)'
 'Credit card (automatic)'])
(Monthly Charges: [ 53.85  70.7   99.65 ... 108.35  63.1   78.7 ])
(Total Charges: [ 108.15  151.65  820.5  ... 7362.9   346.45 6844.5 ])
(Churn Va

female xx thus 0 male xy thus 1


In [9]:
import pickle

# Encode 'Gender' column
gender_mapping = {'Female': 0, 'Male': 1}
df_filtered['Gender'] = df_filtered['Gender'].map(gender_mapping)

# Update encoders dictionary with Gender mapping
# Ensure encoders dictionary is loaded if not in session, or initialized if starting fresh
output_path_encoders = '/content/drive/MyDrive/THESIS/05outputs/encoders.pkl'
try:
    with open(output_path_encoders, 'rb') as f:
        encoders = pickle.load(f)
except FileNotFoundError:
    print(f"Encoders file not found at {output_path_encoders}. Initializing new encoders dictionary.")
    encoders = {}
except Exception as e:
    print(f"Error loading encoders: {e}. Initializing new encoders dictionary.")
    encoders = {}

encoders['Gender'] = gender_mapping

# Save the updated encoders dictionary
with open(output_path_encoders, 'wb') as f:
    pickle.dump(encoders, f)

print("Gender column encoded:")
print(df_filtered['Gender'].value_counts())
print(f"\nUpdated encoders saved to {output_path_encoders}")


Gender column encoded:
Gender
1    3549
0    3483
Name: count, dtype: int64

Updated encoders saved to /content/drive/MyDrive/THESIS/05outputs/encoders.pkl


In [10]:
df_filtered.head()

,CustomerID,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,...,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,0,0,0,2,1,0,DSL,1,...,0,Month-to-month,1,Mailed check,53.85,108.15,1,86,3239,Competitor made better offer
1,9237-HQITU,0,0,0,1,2,1,0,Fiber optic,0,...,0,Month-to-month,1,Electronic check,70.70,151.65,1,67,2701,Moved
2,9305-CDSKC,0,0,0,1,8,1,1,Fiber optic,0,...,1,Month-to-month,1,Electronic check,99.65,820.50,1,86,5372,Moved
3,7892-POOKP,0,0,1,1,28,1,1,Fiber optic,0,...,1,Month-to-month,1,Electronic check,104.80,3046.05,1,84,5003,Moved
4,0280-XJGEX,1,0,0,1,49,1,1,Fiber optic,0,...,1,Month-to-month,1,Bank transfer (automatic),103.70,5036.30,1,89,5340,Competitor had better devices


In [11]:
import pickle
from sklearn.preprocessing import OneHotEncoder

# Define categorical columns for one-hot encoding (excluding binary and target)
categorical_cols_ohe = ['Internet Service', 'Contract', 'Payment Method']

# Identify columns to drop (identifiers and Churn Reason)
drop_cols_before_encoding = ['CustomerID', 'Churn Reason']

# Create a copy to avoid modifying original df_filtered directly before OHE
df_processed = df_filtered.drop(columns=drop_cols_before_encoding, errors='ignore').copy()

# Initialize OneHotEncoder
ohe_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform the selected categorical columns
ohe_results = ohe_encoder.fit_transform(df_processed[categorical_cols_ohe])

# Get feature names for the new one-hot encoded columns
ohe_feature_names = ohe_encoder.get_feature_names_out(categorical_cols_ohe)

# Create a DataFrame from the one-hot encoded results
df_ohe = pd.DataFrame(ohe_results, columns=ohe_feature_names, index=df_processed.index)

# Drop the original categorical columns and concatenate the new one-hot encoded ones
df_processed = df_processed.drop(columns=categorical_cols_ohe)
df_processed = pd.concat([df_processed, df_ohe], axis=1)

print("Shape of df_processed after One-Hot Encoding:")
print(df_processed.shape)

print("\nHead of df_processed showing new one-hot encoded columns:")
display(df_processed.head())

# Load existing encoders (if any), then add the OHE encoder and save
output_path_encoders = '/content/drive/MyDrive/THESIS/05outputs/encoders.pkl'
try:
    with open(output_path_encoders, 'rb') as f:
        encoders = pickle.load(f)
except FileNotFoundError:
    print(f"Encoders file not found at {output_path_encoders}. Initializing new encoders dictionary.")
    encoders = {}
except Exception as e:
    print(f"Error loading encoders: {e}. Initializing new encoders dictionary.")
    encoders = {}

encoders['OneHotEncoder'] = ohe_encoder
encoders['OneHotEncodedColumns'] = categorical_cols_ohe

with open(output_path_encoders, 'wb') as f:
    pickle.dump(encoders, f)

print(f"\nOneHotEncoder and column list saved to {output_path_encoders}")

Shape of df_processed after One-Hot Encoding:
(7032, 29)

Head of df_processed showing new one-hot encoded columns:


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Online Security,Online Backup,Device Protection,...,Internet Service_DSL,Internet Service_Fiber optic,Internet Service_No,Contract_Month-to-month,Contract_One year,Contract_Two year,Payment Method_Bank transfer (automatic),Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,1,0,0,0,2,1,0,1,1,0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0,0,0,1,2,1,0,0,0,0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0,0,0,1,8,1,1,0,0,1,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0,0,1,1,28,1,1,0,0,1,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,1,0,0,1,49,1,1,0,1,1,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0



OneHotEncoder and column list saved to /content/drive/MyDrive/THESIS/05outputs/encoders.pkl


In [12]:
from sklearn.preprocessing import StandardScaler

# Define numerical columns to scale
numerical_cols_to_scale = ['Monthly Charges', 'Total Charges', 'Tenure Months']

# Initialize StandardScaler
scaler = StandardScaler()

# Fit and transform the selected numerical columns
df_processed[numerical_cols_to_scale] = scaler.fit_transform(df_processed[numerical_cols_to_scale])

print("Numerical columns scaled:")
display(df_processed[numerical_cols_to_scale].describe())

# Load existing encoders (if any), then add the StandardScaler and save
output_path_encoders = '/content/drive/MyDrive/THESIS/05outputs/encoders.pkl'
try:
    with open(output_path_encoders, 'rb') as f:
        encoders = pickle.load(f)
except FileNotFoundError:
    print(f"Encoders file not found at {output_path_encoders}. Initializing new encoders dictionary.")
    encoders = {}
except Exception as e:
    print(f"Error loading encoders: {e}. Initializing new encoders dictionary.")
    encoders = {}

encoders['StandardScaler'] = scaler
encoders['ScaledColumns'] = numerical_cols_to_scale

with open(output_path_encoders, 'wb') as f:
    pickle.dump(encoders, f)

print(f"\nStandardScaler and scaled column list saved to {output_path_encoders}")

print("\nHead of df_processed showing scaled numerical columns:")
display(df_processed.head())

Numerical columns scaled:


,Monthly Charges,Total Charges,Tenure Months
count,7.032000e+03,7.032000e+03,7.032000e+03
mean,5.658475e-17,8.891889e-17,-1.293366e-16
std,1.000071e+00,1.000071e+00,1.000071e+00
min,-1.547283e+00,-9.990692e-01,-1.280248e+00
25%,-9.709769e-01,-8.302488e-01,-9.542963e-01
50%,1.845440e-01,-3.908151e-01,-1.394171e-01
75%,8.331482e-01,6.668271e-01,9.199259e-01
max,1.793381e+00,2.824261e+00,1.612573e+00



StandardScaler and scaled column list saved to /content/drive/MyDrive/THESIS/05outputs/encoders.pkl

Head of df_processed showing scaled numerical columns:


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Online Security,Online Backup,Device Protection,...,Internet Service_DSL,Internet Service_Fiber optic,Internet Service_No,Contract_Month-to-month,Contract_One year,Contract_Two year,Payment Method_Bank transfer (automatic),Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,1,0,0,0,-1.239504,1,0,1,1,0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0,0,0,1,-1.239504,1,0,0,0,0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0,0,0,1,-0.995040,1,1,0,0,1,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0,0,1,1,-0.180161,1,1,0,0,1,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,1,0,0,1,0.675462,1,1,0,1,1,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


In [13]:
# Separate target variable (y) from features (X)
y = df_processed['Churn Value']
X = df_processed.drop('Churn Value', axis=1)

print("Shape of X (features):")
print(X.shape)
print("\nShape of y (target):")
print(y.shape)

print("\nHead of X:")
display(X.head())
print("\nHead of y:")
display(y.head())

Shape of X (features):
(7032, 28)

Shape of y (target):
(7032,)

Head of X:


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Online Security,Online Backup,Device Protection,...,Internet Service_DSL,Internet Service_Fiber optic,Internet Service_No,Contract_Month-to-month,Contract_One year,Contract_Two year,Payment Method_Bank transfer (automatic),Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,1,0,0,0,-1.239504,1,0,1,1,0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0,0,0,1,-1.239504,1,0,0,0,0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0,0,0,1,-0.995040,1,1,0,0,1,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0,0,1,1,-0.180161,1,1,0,0,1,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,1,0,0,1,0.675462,1,1,0,1,1,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0



Head of y:


,Churn Value
0,1
1,1
2,1
3,1
4,1


In [14]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Shape of X_train (training features):", X_train.shape)
print("Shape of X_test (testing features):", X_test.shape)
print("Shape of y_train (training target):", y_train.shape)
print("Shape of y_test (testing target):", y_test.shape)

Shape of X_train (training features): (5625, 28)
Shape of X_test (testing features): (1407, 28)
Shape of y_train (training target): (5625,)
Shape of y_test (testing target): (1407,)


In [15]:
len(X_train.columns)

28

In [16]:
from sklearn.model_selection import train_test_split

# Perform a stratified train-test split (70/30)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("Shape of X_dev (development features):", X_dev.shape)
print("Shape of X_test (test features):", X_test.shape)
print("Shape of y_dev (development target):", y_dev.shape)
print("Shape of y_test (test target):", y_test.shape)

Shape of X_dev (development features): (4922, 28)
Shape of X_test (test features): (2110, 28)
Shape of y_dev (development target): (4922,)
Shape of y_test (test target): (2110,)


In [17]:
# Combine X_dev and y_dev to create the development set
dev_set = pd.concat([X_dev, y_dev], axis=1)

# Combine X_test and y_test to create the test set
test_set = pd.concat([X_test, y_test], axis=1)

print("Shape of dev_set:", dev_set.shape)
print("Shape of test_set:", test_set.shape)

Shape of dev_set: (4922, 29)
Shape of test_set: (2110, 29)


In [18]:
output_dir = '/content/drive/MyDrive/THESIS/05outputs/'

# Save dev_set to CSV
dev_set_path = output_dir + 'dev_set.csv'
dev_set.to_csv(dev_set_path, index=False)
print(f"Development set saved to: {dev_set_path}")

# Save test_set to CSV
test_set_path = output_dir + 'test_set.csv'
test_set.to_csv(test_set_path, index=False)
print(f"Test set saved to: {test_set_path}")

Development set saved to: /content/drive/MyDrive/THESIS/05outputs/dev_set.csv
Test set saved to: /content/drive/MyDrive/THESIS/05outputs/test_set.csv


### Churn Distribution in Development (Training) and Test Sets

In [19]:
# Churn distribution in the development set
print("Churn distribution in Development Set (y_dev):")
display(y_dev.value_counts())
display(y_dev.value_counts(normalize=True) * 100)

# Churn distribution in the test set
print("\nChurn distribution in Test Set (y_test):")
display(y_test.value_counts())
display(y_test.value_counts(normalize=True) * 100)

Churn distribution in Development Set (y_dev):


,count
Churn Value,
0,3614
1,1308


,proportion
Churn Value,
0,73.425437
1,26.574563



Churn distribution in Test Set (y_test):


,count
Churn Value,
0,1549
1,561


,proportion
Churn Value,
0,73.412322
1,26.587678
